# NASA Deep Space Image Classification & Neural Training Pipeline

**Author:** Raúl Salas Sahuquillo  
**Repository:** [https://github.com/RaulSalasSahuquillo/nasa-deep-space-classifier](https://github.com/RaulSalasSahuquillo/nasa-deep-space-classifier)  
**License:** Creative Commons Attribution-NonCommercial-ShareAlike 4.0 (CC BY-NC-SA 4.0)  
**Notebook:** `training/dataset_generator.ipynb`  
**Description:** Automated dataset harvester and builder for astronomical targets (NASA Images API & SDSS) with stratified train/test split generation.

In [ ]:
# ==============================================================================
# NASA Deep Space Image Classification & Neural Training Pipeline
# File: training/dataset_generator.ipynb
# Author: Raúl Salas Sahuquillo
# ==============================================================================

import os
import requests
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

# Define base directory structure
base_dir = '/content/data'
categories = ['star', 'galaxy', 'quasar', 'nebula', 'planet']

for category in categories:
    os.makedirs(os.path.join(base_dir, category), exist_ok=True)

print("Directory structure created successfully under /content/data/")

In [ ]:
def download_nasa_highres(query_terms, class_name, limit=1000):
    target_dir = os.path.join(base_dir, class_name)
    existing_files = [f for f in os.listdir(target_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    count = len(existing_files)

    print(f"Collecting high-res images for '{class_name}' (Current: {count}/{limit})...")

    for query in query_terms:
        if count >= limit:
            break

        page = 1
        while count < limit and page <= 50:
            url = f"https://images-api.nasa.gov/search?q={query}&media_type=image&page={page}"
            try:
                res = requests.get(url, timeout=10)
                if res.status_code != 200:
                    break

                items = res.json().get('collection', {}).get('items', [])
                if not items:
                    break

                for item in items:
                    if count >= limit:
                        break
                    links = item.get('links', [])
                    if links:
                        img_url = links[0].get('href')
                        # Upgrade resolution to original/large if available
                        highres_url = img_url.replace('~thumb.jpg', '~orig.jpg').replace('~small.jpg', '~large.jpg')

                        try:
                            img_res = requests.get(highres_url, timeout=6)
                            if img_res.status_code != 200:
                                img_res = requests.get(img_url, timeout=6)  # Fallback to default asset

                            if img_res.status_code == 200 and len(img_res.content) > 5000:
                                save_path = os.path.join(target_dir, f"{class_name}_{count}.jpg")
                                with open(save_path, 'wb') as f:
                                    f.write(img_res.content)
                                count += 1
                        except Exception:
                            continue
                page += 1
            except Exception as e:
                print(f"Network exception for '{query}': {e}")
                break

    print(f"Finished '{class_name}': {count}/{limit} high-res images stored.")

# Alternate SDSS cutout generator to guarantee 1000 items for dense stellar/quasar fields
def download_sdss_cutouts(class_name, limit=1000):
    target_dir = os.path.join(base_dir, class_name)
    existing = len([f for f in os.listdir(target_dir) if f.endswith(('.jpg', '.png'))])
    if existing >= limit:
        return

    print(f"Downloading SDSS astronomical cutouts for '{class_name}'...")
    coords = {'star': (180.0, 0.0), 'quasar': (150.0, 2.0), 'galaxy': (200.0, 15.0)}
    base_ra, base_dec = coords.get(class_name, (180.0, 0.0))

    count = existing
    for i in range(2500):
        if count >= limit:
            break
        ra = base_ra + (i * 0.008)
        dec = base_dec + (i * 0.008)
        # High resolution cutout request (512x512)
        url = f"https://skyserver.sdss.org/dr16/SkyServerWS/ImgViewing/getjpeg?ra={ra}&dec={dec}&scale=0.3&width=512&height=512"
        try:
            r = requests.get(url, timeout=3)
            if r.status_code == 200 and len(r.content) > 3000:
                with open(os.path.join(target_dir, f"sdss_{class_name}_{count}.jpg"), 'wb') as f:
                    f.write(r.content)
                count += 1
        except Exception:
            continue
    print(f"SDSS cutouts for '{class_name}' completed: {count}/{limit} images.")


# BULK DOWNLOAD EXECUTION

# Nebulae
download_nasa_highres(['nebula', 'emission nebula', 'planetary nebula', 'hubble nebula'], 'nebula', limit=1000)

# Planets
download_nasa_highres(['planet', 'jupiter', 'saturn', 'mars', 'neptune'], 'planet', limit=1000)

# Galaxies
download_nasa_highres(['galaxy', 'spiral galaxy', 'elliptical galaxy', 'hubble deep field'], 'galaxy', limit=1000)

# Stars
download_nasa_highres(['star cluster', 'globular cluster', 'pleiades', 'bright star'], 'star', limit=1000)
download_sdss_cutouts('star', limit=1000)

# Quasars
download_nasa_highres(['quasar', 'active galactic nucleus', 'blazar'], 'quasar', limit=1000)
download_sdss_cutouts('quasar', limit=1000)

In [ ]:
print("FINAL DATASET SUMMARY")
total_images = 0

for category in categories:
    cat_dir = os.path.join(base_dir, category)
    num_files = len([f for f in os.listdir(cat_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
    total_images += num_files
    print(f"Category '{category:<8}': {num_files}/1000 images")

print(f"\nTotal Dataset Size: {total_images} / 5000 images ready.")

In [ ]:
class_to_idx = {
    'star': 0,
    'galaxy': 1,
    'quasar': 2,
    'nebula': 3,
    'planet': 4
}

dataset_records = []

for category in categories:
    folder_path = os.path.join(base_dir, category)
    if os.path.exists(folder_path):
        label = class_to_idx[category]
        files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        for file in files:
            full_path = os.path.join(folder_path, file)
            dataset_records.append({'image_path': full_path, 'label': label})

# Create DataFrame
df = pd.DataFrame(dataset_records)

# Perform 80% Train / 20% Test Stratified Split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# Save CSV files to root working directory
train_df.to_csv('/content/cosmos_train.csv', index=False)
test_df.to_csv('/content/cosmos_test.csv', index=False)

print("CSV Manifests Successfully Generated!")
print(f" Train Manifest ('cosmos_train.csv'): {len(train_df)} samples")
print(f" Test Manifest  ('cosmos_test.csv') : {len(test_df)} samples")

In [ ]:
base_dir = '/content/data'
target_count = 1000

# Broad search queries to complete missing categories
search_queries = {
    'star': ['star', 'sun', 'constellation', 'star field', 'milky way stars'],
    'quasar': ['quasar', 'active galactic nucleus', 'blazar', 'deep field quasar', 'chandra space'],
    'nebula': ['nebula', 'orion nebula', 'supernova remnant', 'interstellar dust'],
    'galaxy': ['galaxy', 'hubble galaxy', 'spiral galaxy'],
    'planet': ['planet', 'solar system', 'exoplanet']
}

print("STARTING TOP-UP FOR MISSING DATASET IMAGES\n")

for category, queries in search_queries.items():
    target_dir = os.path.join(base_dir, category)
    os.makedirs(target_dir, exist_ok=True)

    existing_files = [f for f in os.listdir(target_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    count = len(existing_files)

    if count >= target_count:
        print(f"Category '{category:<8}': Already has {count}/{target_count} images. Skipping...")
        continue

    needed = target_count - count
    print(f"Category '{category:<8}': Currently has {count} images. Downloading {needed} more...")

    added = 0
    for query in queries:
        if count >= target_count:
            break

        page = 1
        while count < target_count and page <= 40:
            url = f"https://images-api.nasa.gov/search?q={query}&media_type=image&page={page}"
            try:
                res = requests.get(url, timeout=8)
                if res.status_code != 200:
                    break

                items = res.json().get('collection', {}).get('items', [])
                if not items:
                    break

                for item in items:
                    if count >= target_count:
                        break
                    links = item.get('links', [])
                    if links:
                        img_url = links[0].get('href')
                        try:
                            img_res = requests.get(img_url, timeout=5)
                            if img_res.status_code == 200 and len(img_res.content) > 2500:
                                save_path = os.path.join(target_dir, f"fill_{category}_{count}.jpg")
                                with open(save_path, 'wb') as f:
                                    f.write(img_res.content)
                                count += 1
                                added += 1
                        except Exception:
                            continue
                page += 1
            except Exception as e:
                print(f"Network exception for query '{query}': {e}")
                break

    print(f"Added {added} images to '{category}'. Current total: {count}/{target_count}\n")

print("Dataset top-up completed successfully.")

In [ ]:
class_to_idx = {
    'star': 0,
    'galaxy': 1,
    'quasar': 2,
    'nebula': 3,
    'planet': 4
}

dataset_records = []

for category in ['star', 'galaxy', 'quasar', 'nebula', 'planet']:
    folder_path = os.path.join(base_dir, category)
    if os.path.exists(folder_path):
        label = class_to_idx[category]
        files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        for file in files:
            full_path = os.path.join(folder_path, file)
            dataset_records.append({'image_path': full_path, 'label': label})

# Create updated DataFrame
df = pd.DataFrame(dataset_records)

# Stratified Split 80% Train / 20% Test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# Save updated CSV manifests
train_df.to_csv('/content/cosmos_train.csv', index=False)
test_df.to_csv('/content/cosmos_test.csv', index=False)

print("UPDATED CSV MANIFEST SUMMARY")
print(f"Total Images in Dataset : {len(df)}")
print(f"  Train Manifest ('cosmos_train.csv'): {len(train_df)} samples")
print(f"  Test Manifest  ('cosmos_test.csv') : {len(test_df)} samples")